In [0]:
%sql

-- 0. Create Gold Schema

CREATE SCHEMA IF NOT EXISTS hr_catalog_divansh.gold;


-- Step 20: Payroll summary

CREATE OR REPLACE VIEW hr_catalog_divansh.gold.dept_payroll_summary AS
SELECT
  department_name,
  location,
  COUNT(employee_id) AS employee_count,
  ROUND(SUM(total_monthly_compensation), 2) AS total_monthly_payroll,
  ROUND(AVG(total_monthly_compensation), 2) AS avg_monthly_comp
FROM hr_catalog_divansh.silver.employees_enriched
GROUP BY department_name, location
ORDER BY total_monthly_payroll DESC;


-- Step 21: Headcount vs budget

CREATE OR REPLACE VIEW hr_catalog_divansh.gold.headcount_vs_budget AS
SELECT
  d.department_id,
  d.department_name,
  d.location,
  d.head_count_budget,
  COUNT(e.employee_id) AS actual_headcount,
  COUNT(e.employee_id) - d.head_count_budget AS headcount_variance,
  CASE
    WHEN COUNT(e.employee_id) > d.head_count_budget THEN 'Over Budget'
    WHEN COUNT(e.employee_id) = d.head_count_budget THEN 'At Budget'
    ELSE 'Under Budget'
  END AS budget_status
FROM hr_catalog_divansh.bronze.departments d
LEFT JOIN hr_catalog_divansh.silver.employees_enriched e
  ON d.department_id = e.department_id
GROUP BY
  d.department_id,
  d.department_name,
  d.location,
  d.head_count_budget
ORDER BY headcount_variance DESC;